# Zero, PAD и BOS: полный воспроизводимый пилот

**Один файл → «Выполнить все» → результаты нового обучения, графики и выводы.**

Код проекта встроен в этот ноутбук. Загружать ZIP, подключать Google Drive,
клонировать GitHub или вводить токены не нужно. Данные скачиваются с сайтов
GroupLens и McAuley Lab (около 156 MiB). Нужен доступ в интернет.

По умолчанию достаточно **CPU**. Выполняются 39 основных и 54 контрольных обучения
на MovieLens-1M, Amazon Beauty и Amazon Video Games, seeds 17, 29, 43.
Это бюджет промежуточного отчёта: 3 итерации HOOI и 5 эпох SASRec.
Время зависит от выданной Colab машины и сети; после запуска покажем фактическое.

Сохранённых метрик внутри нет: все таблицы и выводы ниже строятся из этого запуска.
При повторном выполнении ячеек обучения завершённые эксперименты возобновляются;
повторный запуск первой ячейки создаёт новую папку результатов.

## Что проверяем сейчас и что оставляем на следующий этап

**Сейчас:** H1 — меняют ли PAD/BOS качество относительно Zero;
H2 — дешевле ли BOS при сопоставимом качестве с PAD;
H3 — зависит ли эффект от доступной длины истории;
H4 — сохраняется ли эффект при изменении веса служебных наблюдений и нормализации.
H4 — диагностические контроли, а не полное установление причины улучшения.

**Далее:** сохраняется ли выигрыш после равного подбора гиперпараметров и обучения
SASRec до сходимости; устойчив ли он к K, локальному окну, seed и временным границам;
даёт ли сама граница истории эффект при уравненном вкладе историй после Hankel
и при раздельном включении служебного вклада в обучение и предсказание.

## 1. Установка и встроенный код

Все исходники и 12 проверок корректности будут сохранены в рабочую папку.
Уже подходящие версии библиотек сохраняются; точное окружение записывается в результаты.
`SASREC_DEVICE="cpu"` соответствует режиму отчёта. Значение `auto` использует CUDA,
если она доступна; измерения такого SASRec нельзя смешивать с CPU-временем отчёта.

In [ ]:
# @title Настройка запуска и установка зависимостей
SASREC_DEVICE = "cpu"  # @param ["cpu", "auto"]
import os, sys, json, hashlib, subprocess, time, platform, zipfile, socket
from pathlib import Path
from datetime import datetime, timezone
NOTEBOOK_STARTED = time.perf_counter()
socket.setdefaulttimeout(60)
BASE = Path("/content") if Path("/content").is_dir() else Path.cwd()
PROJECT = BASE / "satf_padding_single"
PROJECT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT)
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[key] = "2"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy>=1.26,<3", "scipy>=1.12,<2", "pandas>=2.1,<3", "torch>=2.2,<3",
                "psutil>=5.9", "matplotlib>=3.8,<4", "pytest>=8"], check=True)
print("Рабочая папка:", PROJECT)


In [ ]:
# @title Встроенные исходники проекта (можно раскрыть)
PROJECT_FILES = {'pyproject.toml': '[build-system]\nrequires = ["setuptools>=68"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "satf-padding-bos"\nversion = "0.1.0"\ndescription = "Reproducible Zero/PAD/BOS experiments for locally attentive sequential tensor factorization"\nrequires-python = ">=3.10"\ndependencies = ["numpy>=1.26,<3", "scipy>=1.12,<2", "pandas>=2.1,<3", "torch>=2.2,<3", "psutil>=5.9", "matplotlib>=3.8,<4"]\n\n[project.optional-dependencies]\ndev = ["pytest>=8", "nbformat>=5", "nbclient>=0.10", "ipykernel>=6"]\nreport = ["reportlab>=4", "pypdf>=5", "pillow>=10"]\n\n[tool.setuptools.packages.find]\nwhere = ["src"]\n\n[tool.pytest.ini_options]\npythonpath = ["src"]\ntestpaths = ["tests"]\n', 'docs/interim/protocol.md': '# Протокол эксперимента v1\n\n## Предобработка и время\n\nДатасеты: MovieLens-1M, Amazon Beauty 2014 5-core, Amazon Video Games 2014 5-core. Любая оценка 1–5 означает факт взаимодействия, порог по рейтингу не применяется. Предсказывается следующее взаимодействие, а не положительное отношение. Используются только ID и timestamp; тексты, изображения и демография не используются.\n\nСобытия сортируются по `(timestamp, user_id, item_id)` со стабильной сортировкой. Для пары `(user,item)` сохраняется самое раннее событие. Повторы затем не являются целями. При одинаковом timestamp порядок по строковому ID условный, не восстановленный реальный порядок. Новая k-core-фильтрация не проводится. Предварительный 5-core Amazon построен владельцем по всей выборке: это ограничение отбора популяции.\n\n`t1 = lower_quantile(timestamp, 0.8)`, `t2 = lower_quantile(timestamp, 0.9)` вычисляются после дедупликации. Train: `t < t1`; validation: `t1 <= t < t2`; test: `t >= t2`. Все равные timestamp остаются на одной стороне границы. Это приблизительные доли событий, не индивидуальный leave-last-out.\n\nКаталог и словарь объектов строятся только из train. Модели обучаются один раз на train и не дообучаются на validation. На validation для каждого пользователя берётся первое новое событие интервала, история — события до t1. На test берётся первое новое событие после t2, история — события до t2, включая validation. Для этого первого события история совпадает с доступной до момента предсказания. В истории остаются известные train-каталогу объекты. Cold-target исключается, но НЕ заменяется следующим warm-target. Отдельно считаются исключения cold-target и отсутствие известной истории; счётчики последовательные и не перекрываются.\n\nДлина доступной модельной истории — число известных к моменту предсказания объектов, обрезанное до K. При оценке из полного train-каталога исключается ВСЯ доступная история, а не только хвост K. Служебные объекты никогда не попадают в кандидаты. Равные score разрешаются по возрастанию индекса объекта; индекс задан строковой сортировкой train-ID.\n\n## Кодировки и инференс\n\nПри n<K: Zero = q нулей + история; PAD = q наблюдений одного служебного объекта + история; BOS = q−1 нулей + одна метка + история. q=K−n. При n>=K берётся хвост K без служебных токенов. Сначала строится кодировка длины K, затем для прогнозирования она сдвигается на один шаг влево, последняя позиция остаётся неизвестной. После сдвига новый BOS не добавляется. Это сохраняет одинаковые входы Zero/PAD/BOS при полной истории.\n\nВ HOOI всегда резервируется одинаковая дополнительная строка каталога, даже для Zero, где она нулевая. В обучении одна последовательность на пользователя. Hankel длины K имеет размеры L×(K−L+1), элемент (a,b) равен x[a+b]. Свёртки выполняются через K разреженных пользователь–объект матриц; все позиции повторяющегося PAD учитываются. Полный четырёхмерный тензор создаётся только в тестах.\n\nОсновная конфигурация: K=50, L=5, ранги (12,12,2,2), 3 итерации HOOI, нижнетреугольное A[a,b]=(a−b+1)^−1 при a>=b. Нормировка частоты выключена (scaling=1). Все три кодировки имеют одинаковые начальные матрицы при одном seed. Включение нормировки задаётся весом объекта max(count,1)^((scaling−1)/2), count считается по реальным train-хвостам. Служебный вес по умолчанию равен 1. На выходе обратное масштабирование не применяется.\n\nSASRec: собственная PyTorch-реализация causal attention, 1 блок и 1 голова, dim=32, dropout=0.2, Adam lr=0.001, betas=(0.9,0.98), batch=128, 5 эпох. Обучение на последних K пар next-item каждого train-пользователя с >=2 событиями. BCE с одним равномерным отрицательным объектом на положительный, исключены только объекты train-истории пользователя. У модели нет доступа к будущим validation/test-положительным объектам при sampling. MostPopular считает все реальные train-взаимодействия; это диагностический бейзлайн без настройки.\n\n## Бюджет, метрики и неопределённость\n\nПилот использует одну заранее зафиксированную конфигурацию на модель, без выбора по test и без заявления об оптимальности. Seeds: 17,29,43. MostPopular детерминирован и запускается один раз. Между семействами не уравнивается wall-clock: 3 шага HOOI и 5 эпох SASRec не равнозначны степени сходимости. Значения validation сохраняются для будущего подбора, текущая конфигурация по ним не выбиралась.\n\nОдин target на пользователя. NDCG@10 = 1/log2(rank+1), если rank<=10, иначе 0. HR@10 = 1[rank<=10]. Оценка по полному допустимому каталогу, без negative sampling. Среднее по пользователям, затем среднее и SD по seed. Отдельные срезы: n<K, n=K и интервалы 1–5,6–20,21+ (n вычислено в момент предсказания).\n\nПарный bootstrap: усреднить NDCG каждого пользователя по трём seed, затем 2000 раз переотобрать пользователей с возвращением; показать percentile 95% CI разности с Zero. Интервал условен на этих seed и не покрывает полностью неопределённость обучения. Три запуска не позволяют надёжно оценить её хвосты. При множественных контрастах не делается заявление о подтверждённой статистической значимости.\n\nВремя обучения включает построение представления, fit и подготовку проектора; загрузка/предобработка данных исключены. Время оценки включает score, маскирование и вычисление ранга. Peak RSS — максимум памяти всего отдельного процесса, семплируемый каждые 20 мс во время fit+eval; это НЕ размер модели и НЕ GPU VRAM. Для текущих измерений модели запускаются последовательно на CPU, 2 вычислительных потока. Colab GPU-запуски записываются в отдельную папку и не смешиваются с CPU-временем.\n\n## Диагностика механизма\n\n`configs/ablations.json`: PAD/sqrt(q) уравнивает суммарный квадрат веса служебных записей в исходном позиционном тензоре (но не после Hankel/A). Leftmost marker ставится в нулевую позицию пустого префикса и удаляется при сдвиге инференса; это контроль влияния служебного наблюдения в обучении. Варианты scale=0.2 сравнивают Zero/PAD/BOS при нормировке реальных объектов, а PAD frequency дополнительно нормирует служебный токен по его train-частоте. Эти контрасты диагностические: они не являются полной причинной идентификацией отдельного механизма.\n', 'src/satf_padding/__init__.py': '"""Independent research implementation of LA-SATF and its padding variants."""\n__version__ = "0.1.0"\n', 'src/satf_padding/data.py': '"""Train-only vocabulary and global chronological evaluation protocol."""\nfrom pathlib import Path\nimport gzip\nimport hashlib\nimport json\nimport urllib.request\nimport zipfile\nimport numpy as np\nimport pandas as pd\n\nSOURCES = {\n    "ml1m": ("ml-1m.zip", "https://files.grouplens.org/datasets/movielens/ml-1m.zip"),\n    "beauty": ("reviews_Beauty_5.json.gz", "https://mcauleylab.ucsd.edu/public_datasets/data/amazon/categoryFiles/reviews_Beauty_5.json.gz"),\n    "games": ("reviews_Video_Games_5.json.gz", "https://mcauleylab.ucsd.edu/public_datasets/data/amazon/categoryFiles/reviews_Video_Games_5.json.gz"),\n}\n\ndef download(name, root="data/raw"):\n    filename, url = SOURCES[name]\n    path = Path(root) / filename\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if not path.exists():\n        temp = path.with_suffix(path.suffix + ".part")\n        urllib.request.urlretrieve(url, temp)\n        temp.replace(path)\n    return path\n\ndef load(name, root="data/raw"):\n    path = download(name, root)\n    if name == "ml1m":\n        with zipfile.ZipFile(path) as z:\n            with z.open("ml-1m/ratings.dat") as f:\n                df = pd.read_csv(f, sep="::", engine="python", names=["user", "item", "rating", "time"])\n    else:\n        with gzip.open(path, "rt") as f:\n            records = (json.loads(line) for line in f)\n            df = pd.DataFrame((r["reviewerID"], r["asin"], r["overall"], r["unixReviewTime"]) for r in records)\n        df.columns = ["user", "item", "rating", "time"]\n    df["user"] = df.user.astype(str)\n    df["item"] = df.item.astype(str)\n    return df, hashlib.sha256(path.read_bytes()).hexdigest()\n\ndef clean(df):\n    """Every recorded rating is an implicit event. Earliest pair wins."""\n    df = df[["user", "item", "time"]].dropna().copy()\n    df["time"] = df.time.astype("int64")\n    return df.sort_values(["time", "user", "item"], kind="stable").drop_duplicates(["user", "item"], keep="first").reset_index(drop=True)\n\ndef prepare(df, quantiles=(.8, .9)):\n    raw_n = len(df)\n    df = clean(df)\n    t1, t2 = (int(df.time.quantile(q, interpolation="lower")) for q in quantiles)\n    if t1 >= t2:\n        raise ValueError("Need distinct temporal boundaries")\n    tr = df[df.time < t1]\n    items = sorted(tr.item.unique())\n    mapping = {x: i for i, x in enumerate(items)}\n    histories = [g.item.map(mapping).to_numpy(dtype=int) for _, g in tr.groupby("user", sort=True)]\n    samples, diagnostics = {}, {}\n    for label, lo, hi in [("validation", t1, t2), ("test", t2, None)]:\n        interval = df[df.time >= lo]\n        if hi is not None:\n            interval = interval[interval.time < hi]\n        # First next-new event, never replace a cold target with a later warm one.\n        targets = interval.groupby("user", sort=True).head(1)\n        before = df[df.time < lo]\n        history_map = {u: g.item.tolist() for u, g in before.groupby("user", sort=False)}\n        rows, cold, no_history = [], 0, 0\n        for r in targets.itertuples():\n            hist = [mapping[x] for x in history_map.get(r.user, []) if x in mapping]\n            if r.item not in mapping:\n                cold += 1\n                continue\n            if not hist:\n                no_history += 1\n                continue\n            rows.append({"user": r.user, "history": hist, "target": mapping[r.item], "time": int(r.time)})\n        samples[label] = rows\n        diagnostics[label] = {"targets": len(targets), "evaluated": len(rows), "cold_target": cold, "empty_known_history": no_history}\n    split_payload = {"train": tr.to_csv(index=False), "v": samples["validation"], "t": samples["test"]}\n    split_hash = hashlib.sha256(json.dumps(split_payload, sort_keys=True).encode()).hexdigest()\n    stats = {"raw_events": raw_n, "events": len(df), "users": int(df.user.nunique()), "items": int(df.item.nunique()), "train_events": len(tr), "train_users": len(histories), "catalog": len(items), "t1": t1, "t2": t2, "split_sha256": split_hash, **diagnostics}\n    return {"histories": histories, "samples": samples, "n_items": len(items), "stats": stats}\n', 'src/satf_padding/evaluation.py': 'import time\nimport numpy as np\n\ndef target_rank(scores, history, target):\n    scores = np.asarray(scores, dtype=float).copy()\n    scores[np.asarray(history, int)] = -np.inf\n    if target in history or not np.isfinite(scores[target]):\n        raise ValueError("Invalid target or nonfinite prediction")\n    # Stable full-catalog ordering: score desc, item index asc.\n    rank = 1 + np.sum(scores > scores[target]) + np.sum((scores == scores[target]) & (np.arange(len(scores)) < target))\n    return int(rank)\n\ndef evaluate(model, rows, k=50, batch_size=128):\n    ranks, lengths = [], []\n    start = time.perf_counter()\n    for offset in range(0, len(rows), batch_size):\n        batch = rows[offset:offset+batch_size]\n        scores = model.score_batch([r["history"] for r in batch])\n        for r, score in zip(batch, scores):\n            ranks.append(target_rank(score, r["history"], r["target"]))\n            lengths.append(min(len(r["history"]), k))\n    elapsed = time.perf_counter() - start\n    ranks, lengths = np.asarray(ranks), np.asarray(lengths)\n    ndcg = np.where(ranks <= 10, 1. / np.log2(ranks + 1), 0.)\n    hit = (ranks <= 10).astype(float)\n    metrics = {"n": len(rows), "ndcg": float(ndcg.mean()), "hr": float(hit.mean()), "eval_seconds": elapsed, "ms_per_user": elapsed*1000/max(len(rows),1)}\n    for label, mask in [("short", lengths < k), ("full", lengths == k), ("1_5", lengths <= 5), ("6_20", (lengths > 5) & (lengths <= 20)), ("21_plus", lengths > 20)]:\n        metrics[label+"_n"] = int(mask.sum())\n        metrics[label+"_ndcg"] = float(ndcg[mask].mean()) if mask.any() else None\n    return metrics, {"ranks": ranks, "lengths": lengths, "ndcg": ndcg}\n', 'src/satf_padding/experiment.py': '"""Run each training seed in a fresh process; resume only identical manifests."""\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport pickle\nimport platform\nimport subprocess\nimport sys\nimport threading\nimport time\nimport numpy as np\nimport psutil\nfrom .data import load, prepare\nfrom .evaluation import evaluate\nfrom .tensor import LASATF, MostPopular\n\ndef source_hash():\n    h = hashlib.sha256()\n    for p in sorted(Path(__file__).parent.glob("*.py")):\n        h.update(p.name.encode()); h.update(p.read_bytes())\n    return h.hexdigest()\n\ndef trial(args):\n    spec = json.loads(args.trial)\n    data = pickle.loads(Path(spec["cache"]).read_bytes())\n    cfg, name, seed = spec["config"], spec["model"], spec["seed"]\n    if name == "popular":\n        model = MostPopular(data["n_items"])\n    elif name == "sasrec":\n        import torch\n        torch.set_num_threads(spec["threads"])\n        from .sasrec import SASRec\n        model = SASRec(data["n_items"], k=cfg["k"], seed=seed, device=spec["device"], **cfg["sasrec"])\n    else:\n        params = dict(cfg["tensor"])\n        params.update(mode=name.split("_")[0], balance="balanced" in name, special_scaling="frequency" if "frequency" in name else "unit")\n        if "scaled" in name:\n            params["scaling"] = .2\n        model = LASATF(data["n_items"], k=cfg["k"], seed=seed, **params)\n    running = threading.Event(); running.set()\n    rss = [psutil.Process().memory_info().rss]\n    def monitor():\n        while running.is_set():\n            rss[0] = max(rss[0], psutil.Process().memory_info().rss)\n            time.sleep(.02)\n    thread = threading.Thread(target=monitor, daemon=True); thread.start()\n    start = time.perf_counter()\n    model.fit(data["histories"])\n    fit_seconds = time.perf_counter() - start\n    results = {}\n    out = Path(spec["output"])\n    out.parent.mkdir(parents=True, exist_ok=True)\n    for split in ("validation", "test"):\n        metrics, arrays = evaluate(model, data["samples"][split], cfg["k"])\n        results[split] = metrics\n        np.savez_compressed(out.with_name(out.stem + "_" + split + ".npz"), **arrays)\n    running.clear(); thread.join()\n    results.update({"dataset": spec["dataset"], "model": name, "seed": seed, "fit_seconds": fit_seconds, "peak_rss_mb": rss[0]/2**20, "nnz": getattr(model, "nnz", None), "losses": getattr(model, "losses", None), "core_norms": getattr(model, "core_norms", None), "source_sha256": spec["source_sha256"], "config": cfg, "split_sha256": data["stats"]["split_sha256"], "device": str(getattr(model, "device", "cpu")), "threads": spec["threads"], "platform": platform.platform(), "python": platform.python_version()})\n    out.write_text(json.dumps(results, indent=2, allow_nan=False))\n    print(f"{spec[\'dataset\']} {name} seed={seed}: NDCG={results[\'test\'][\'ndcg\']:.5f}, train={fit_seconds:.1f}s", flush=True)\n\ndef run(args):\n    cfg = json.loads(Path(args.config).read_text())\n    out = Path(args.output); out.mkdir(parents=True, exist_ok=True)\n    code_sha = source_hash()\n    for dataset in (args.datasets.split(",") if args.datasets else cfg["datasets"]):\n        raw, raw_sha = load(dataset, args.raw)\n        data = prepare(raw, cfg["quantiles"])\n        data["stats"]["raw_sha256"] = raw_sha\n        (out / (dataset + "_data.json")).write_text(json.dumps(data["stats"], indent=2))\n        cache = Path("data/processed") / (dataset + "_" + data["stats"]["split_sha256"][:12] + ".pkl")\n        cache.parent.mkdir(parents=True, exist_ok=True); cache.write_bytes(pickle.dumps(data))\n        models = args.models.split(",") if args.models else cfg["models"]\n        for name in models:\n            for seed in (cfg["seeds"][:1] if name == "popular" else cfg["seeds"]):\n                target = out / f"{dataset}_{name}_{seed}.json"\n                if target.exists() and not args.force:\n                    old = json.loads(target.read_text())\n                    if old["source_sha256"] == code_sha and old["config"] == cfg and old["split_sha256"] == data["stats"]["split_sha256"]:\n                        print("Already complete:", target, flush=True); continue\n                    raise RuntimeError(f"Stale output {target}; choose a new output folder or --force")\n                spec = {"config": cfg, "dataset": dataset, "model": name, "seed": seed, "cache": str(cache), "output": str(target), "source_sha256": code_sha, "threads": args.threads, "device": args.device}\n                env = dict(os.environ, OMP_NUM_THREADS=str(args.threads), OPENBLAS_NUM_THREADS=str(args.threads), MKL_NUM_THREADS=str(args.threads), VECLIB_MAXIMUM_THREADS=str(args.threads), CUBLAS_WORKSPACE_CONFIG=":4096:8")\n                subprocess.run([sys.executable, "-m", "satf_padding.experiment", "--trial", json.dumps(spec)], env=env, check=True)\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--config", default="configs/interim.json"); p.add_argument("--output", default="results/interim")\n    p.add_argument("--raw", default="data/raw"); p.add_argument("--datasets"); p.add_argument("--models")\n    p.add_argument("--threads", type=int, default=2); p.add_argument("--device", default="cpu", choices=["cpu", "cuda", "auto"])\n    p.add_argument("--force", action="store_true"); p.add_argument("--trial")\n    args = p.parse_args()\n    trial(args) if args.trial else run(args)\n\nif __name__ == "__main__":\n    main()\n', 'src/satf_padding/sasrec.py': '"""Compact PyTorch SASRec baseline: causal attention and sampled BCE.\n\nArchitecture follows Kang & McAuley (ICDM 2018); this is our own implementation,\nnot an assertion of exact reproduction of the original paper\'s scores.\n"""\nimport random\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nclass Block(nn.Module):\n    def __init__(self, dim, dropout):\n        super().__init__()\n        self.ln1 = nn.LayerNorm(dim)\n        self.attn = nn.MultiheadAttention(dim, 1, dropout=dropout, batch_first=True)\n        self.ln2 = nn.LayerNorm(dim)\n        self.ff = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(dim, dim), nn.Dropout(dropout))\n    def forward(self, x, causal, nonzero):\n        q = self.ln1(x)\n        a, _ = self.attn(q, x, x, attn_mask=causal, need_weights=False)\n        x = (q + a) * nonzero\n        x = (x + self.ff(self.ln2(x))) * nonzero\n        return x\n\nclass Network(nn.Module):\n    def __init__(self, n_items, k, dim, blocks, dropout):\n        super().__init__()\n        self.items = nn.Embedding(n_items+1, dim, padding_idx=0)\n        self.positions = nn.Embedding(k, dim)\n        self.blocks = nn.ModuleList([Block(dim, dropout) for _ in range(blocks)])\n        self.dropout = nn.Dropout(dropout)\n        self.norm = nn.LayerNorm(dim)\n        self.dim, self.k = dim, k\n    def forward(self, ids):\n        valid = (ids != 0).unsqueeze(-1)\n        x = self.items(ids) * self.dim ** .5\n        x = self.dropout(x + self.positions(torch.arange(self.k, device=ids.device))) * valid\n        causal = torch.ones((self.k, self.k), device=ids.device, dtype=torch.bool).triu(1)\n        for block in self.blocks:\n            x = block(x, causal, valid)\n        return self.norm(x) * valid\n\nclass SASRec:\n    def __init__(self, n_items, k=50, dim=32, blocks=1, dropout=.2, lr=.001, epochs=5, batch_size=128, seed=0, device="cpu"):\n        self.n_items, self.k, self.epochs, self.batch_size = n_items, k, epochs, batch_size\n        self.device = torch.device("cuda" if device == "auto" and torch.cuda.is_available() else "cpu" if device == "auto" else device)\n        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n        if torch.cuda.is_available():\n            torch.cuda.manual_seed_all(seed)\n        torch.use_deterministic_algorithms(True, warn_only=True)\n        self.rng = np.random.default_rng(seed)\n        self.net = Network(n_items, k, dim, blocks, dropout).to(self.device)\n        self.optimizer = torch.optim.Adam(self.net.parameters(), lr=lr, betas=(.9, .98))\n        self.losses = []\n\n    def fit(self, histories):\n        examples = [np.asarray(h, int) for h in histories if 2 <= len(h) < self.n_items]\n        if not examples:\n            raise ValueError("No train sequences with positive and negative items")\n        self.net.train()\n        for _ in range(self.epochs):\n            total, count = 0., 0\n            for start in range(0, len(examples), self.batch_size):\n                # Permute once per epoch, retaining all examples exactly once.\n                if start == 0:\n                    order = self.rng.permutation(len(examples))\n                selected = [examples[i] for i in order[start:start+self.batch_size]]\n                xs = np.zeros((len(selected), self.k), dtype=np.int64)\n                ys = np.zeros_like(xs); ns = np.zeros_like(xs)\n                for row, h in enumerate(selected):\n                    x, y = h[:-1][-self.k:] + 1, h[1:][-self.k:] + 1\n                    xs[row, -len(x):], ys[row, -len(y):] = x, y\n                    neg = self.rng.integers(0, self.n_items, len(y))\n                    bad = np.isin(neg, h)\n                    while bad.any():\n                        neg[bad] = self.rng.integers(0, self.n_items, bad.sum())\n                        bad = np.isin(neg, h)\n                    ns[row, -len(y):] = neg + 1\n                x, y, n = (torch.as_tensor(a, device=self.device) for a in (xs, ys, ns))\n                features = self.net(x)\n                positive = (features * self.net.items(y)).sum(-1)\n                negative = (features * self.net.items(n)).sum(-1)\n                mask = y != 0\n                loss = (F.softplus(-positive[mask]) + F.softplus(negative[mask])).mean()\n                self.optimizer.zero_grad(); loss.backward(); self.optimizer.step()\n                total += loss.item() * mask.sum().item(); count += mask.sum().item()\n            self.losses.append(total / count)\n        self.net.eval()\n        return self\n\n    def score_batch(self, histories):\n        x = np.zeros((len(histories), self.k), dtype=np.int64)\n        for row, h in enumerate(histories):\n            h = np.asarray(h)[-self.k:] + 1\n            x[row, -len(h):] = h\n        with torch.no_grad():\n            z = self.net(torch.as_tensor(x, device=self.device))[:, -1]\n            scores = z @ self.net.items.weight[1:].T\n            return scores.cpu().numpy()\n', 'src/satf_padding/tensor.py': '"""LA-SATF HOOI using sparse position slices, without a dense 4-D tensor.\n\nIndependent implementation of Frolov & Oseledets (2023), Algorithms 2,\nEqs. 18-21. Each positional slice sums all observations: repeated PAD is\nnot collapsed into a single user-item observation.\n"""\nimport numpy as np\nfrom scipy import sparse\nfrom scipy.linalg import eigh, solve_triangular\n\ndef encode(history, k, mode="zero", special=None, amplitude=1.0, balance=False):\n    if mode not in {"zero", "pad", "bos", "start"}:\n        raise ValueError(mode)\n    history = np.asarray(history, dtype=int)[-k:]\n    n = len(history)\n    q = k - n\n    pos, items, val = list(range(q, k)), history.tolist(), [1.] * n\n    if q and mode != "zero":\n        if special is None:\n            raise ValueError("special id required")\n        prefix = list(range(q)) if mode == "pad" else [q - 1] if mode == "bos" else [0]\n        a = amplitude / np.sqrt(len(prefix)) if balance else amplitude\n        pos += prefix\n        items += [special] * len(prefix)\n        val += [a] * len(prefix)\n    return np.asarray(items, int), np.asarray(pos, int), np.asarray(val, float)\n\ndef position_slices(histories, n_items, k, mode="zero", amplitude=1., balance=False):\n    users, items, positions, values = [], [], [], []\n    for u, history in enumerate(histories):\n        ids, p, v = encode(history, k, mode, n_items, amplitude, balance)\n        users.extend([u] * len(p)); items.extend(ids); positions.extend(p); values.extend(v)\n    users, items, positions, values = map(np.asarray, (users, items, positions, values))\n    # Always reserve the same zero/special row to keep dimensions/initialization equal.\n    shape = (len(histories), n_items + 1)\n    result = []\n    for p in range(k):\n        mask = positions == p\n        result.append(sparse.coo_matrix((values[mask], (users[mask], items[mask])), shape=shape).tocsr())\n    return result\n\ndef attention_matrix(window, decay):\n    lag = np.arange(window)[:, None] - np.arange(window)[None, :]\n    return np.where(lag >= 0, (np.maximum(lag, 0) + 1.) ** (-decay), 0.)\n\ndef hankel_weights(a, b):\n    result = np.zeros((len(a) + len(b) - 1, a.shape[1], b.shape[1]))\n    for i in range(a.shape[1]):\n        for j in range(b.shape[1]):\n            result[:, i, j] = np.convolve(a[:, i], b[:, j])\n    return result\n\ndef leading_left(matrix, rank):\n    if rank > min(matrix.shape):\n        raise ValueError(f"Rank {rank} exceeds compressed unfolding {matrix.shape}")\n    # Small Gram matrix + QR avoids expensive SVD of a tall unfolding.\n    if matrix.shape[0] <= matrix.shape[1]:\n        _, vec = eigh(matrix @ matrix.T, subset_by_index=[matrix.shape[0]-rank, matrix.shape[0]-1])\n        return vec[:, ::-1]\n    _, right = eigh(matrix.T @ matrix, subset_by_index=[matrix.shape[1]-rank, matrix.shape[1]-1])\n    return np.linalg.qr(matrix @ right[:, ::-1], mode="reduced")[0]\n\ndef contract(slices, factors, attention, mode):\n    u, v, wl, ws = factors\n    wa = attention @ wl\n    if mode in (0, 1):\n        h = hankel_weights(wa, ws)\n        other = v if mode == 0 else u\n        size = slices[0].shape[mode]\n        out = np.zeros((size, other.shape[1], wl.shape[1], ws.shape[1]))\n        for p, x in enumerate(slices):\n            z = x @ other if mode == 0 else x.T @ other\n            out += np.einsum("ir,ab->irab", z, h[p], optimize=True)\n        return out.reshape(size, -1)\n    c = np.stack([u.T @ (x @ v) for x in slices])\n    if mode == 2:\n        out = np.stack([np.einsum("suv,sr->uvr", c[a:a+len(ws)], ws) for a in range(len(wl))])\n        return (attention.T @ out.reshape(len(wl), -1))\n    out = np.stack([np.einsum("auv,ar->uvr", c[s:s+len(wl)], wa) for s in range(len(ws))])\n    return out.reshape(len(ws), -1)\n\nclass LASATF:\n    def __init__(self, n_items, k=50, window=5, rank=12, local_rank=2, sequence_rank=2, decay=1., scaling=1., iterations=3, mode="zero", seed=0, amplitude=1., balance=False, special_scaling="unit"):\n        self.n_items, self.k, self.mode = n_items, k, mode\n        self.window, self.rank, self.local_rank, self.sequence_rank = window, rank, local_rank, sequence_rank\n        self.decay, self.scaling, self.iterations, self.seed = decay, scaling, iterations, seed\n        self.amplitude, self.balance, self.special_scaling = amplitude, balance, special_scaling\n\n    def fit(self, histories):\n        if not 1 <= self.window <= self.k:\n            raise ValueError("window must lie in [1,K]")\n        slices = position_slices(histories, self.n_items, self.k, self.mode, self.amplitude, self.balance)\n        self.nnz = sum(s.nnz for s in slices)\n        # Popularities use only real train tails. Service row is unit-weight by default.\n        counts = np.bincount(np.concatenate([h[-self.k:] for h in histories]), minlength=self.n_items + 1).astype(float)\n        if self.special_scaling == "frequency":\n            counts[-1] = sum(x[:, -1].count_nonzero() for x in slices)\n        elif self.special_scaling != "unit":\n            raise ValueError(self.special_scaling)\n        self.weights = np.maximum(counts, 1.) ** ((self.scaling - 1.) / 2.)\n        if self.special_scaling == "unit":\n            self.weights[-1] = 1.\n        slices = [x.multiply(self.weights).tocsr() for x in slices]\n        self.attention = attention_matrix(self.window, self.decay)\n        dims = (len(histories), self.n_items + 1, self.window, self.k-self.window+1)\n        ranks = (self.rank, self.rank, self.local_rank, self.sequence_rank)\n        rng = np.random.default_rng(self.seed)\n        self.factors = [np.linalg.qr(rng.normal(size=(d, r)), mode="reduced")[0] for d, r in zip(dims, ranks)]\n        if any(r > d for d, r in zip(dims, ranks)):\n            raise ValueError("Rank larger than a dimension")\n        self.core_norms = []\n        for _ in range(self.iterations):\n            for mode, rank in enumerate(ranks):\n                mat = contract(slices, self.factors, self.attention, mode)\n                self.factors[mode] = leading_left(mat, rank)\n            self.core_norms.append(float(np.linalg.norm(self.factors[3].T @ mat)))\n        _, self.v, wl, ws = self.factors\n        inverse_w = solve_triangular(self.attention.T, wl, lower=False)\n        left = (self.attention @ wl) @ inverse_w[-1]\n        right = ws @ ws[-1]\n        self.prediction_weights = np.convolve(left, right)\n        return self\n\n    def score_batch(self, histories):\n        latent = np.zeros((len(histories), self.rank))\n        for row, h in enumerate(histories):\n            ids, p, vals = encode(h, self.k, self.mode, self.n_items, self.amplitude, self.balance)\n            # Encode first, then shift left: do not insert a new BOS after shifting.\n            take = p > 0\n            ids, p, vals = ids[take], p[take] - 1, vals[take]\n            latent[row] = (self.v[ids] * (vals * self.weights[ids] * self.prediction_weights[p])[:, None]).sum(axis=0)\n        # Score only real items. No inverse-popularity rescaling at output.\n        return latent @ self.v[:self.n_items].T\n\nclass MostPopular:\n    def __init__(self, n_items):\n        self.n_items = n_items\n    def fit(self, histories):\n        self.counts = np.bincount(np.concatenate(histories), minlength=self.n_items).astype(float)\n        return self\n    def score_batch(self, histories):\n        return np.tile(self.counts, (len(histories), 1))\n', 'tests/test_protocol.py': 'import numpy as np\nimport pandas as pd\nfrom satf_padding.data import clean, prepare\nfrom satf_padding.evaluation import evaluate\nfrom satf_padding.tensor import MostPopular\n\ndef test_global_cutoff_dedup_and_no_future_catalog():\n    rows = [(str(u), str(i % 8), i+u, 1.) for u in range(5) for i in range(8)]\n    rows += [("0", "0", 50, 5.), ("0", "future", 40, 5.)]\n    df = pd.DataFrame(rows, columns=["user", "item", "time", "rating"])\n    c = clean(df)\n    assert len(c.query(\'user == "0" and item == "0"\')) == 1\n    assert c.query(\'user == "0" and item == "0"\').time.iloc[0] == 0\n    data = prepare(df, (.6, .8))\n    assert data["n_items"] == c[c.time < data["stats"]["t1"]].item.nunique()\n    assert data["stats"]["test"]["cold_target"] >= 1\n    for split, rows in data["samples"].items():\n        for row in rows:\n            assert row["target"] not in row["history"]\n    assert prepare(df, (.6, .8))["stats"]["split_sha256"] == data["stats"]["split_sha256"]\n\ndef test_ndcg_known_ranks():\n    m = MostPopular(4).fit([np.array([0, 1, 1, 2])])\n    metrics, _ = evaluate(m, [{"history": [0], "target": 1}, {"history": [1], "target": 2}], k=3)\n    assert np.isclose(metrics["ndcg"], (1 + 1/np.log2(3))/2)\n', 'tests/test_sasrec.py': 'import numpy as np\nimport torch\nfrom satf_padding.sasrec import Network, SASRec\n\ndef test_attention_is_causal_and_padding_is_finite():\n    torch.manual_seed(3)\n    net = Network(7, 5, 8, 1, 0.).eval()\n    a = torch.tensor([[0, 0, 1, 2, 3]])\n    b = torch.tensor([[0, 0, 1, 2, 5]])\n    with torch.no_grad():\n        x, y = net(a), net(b)\n    assert torch.isfinite(x).all()\n    torch.testing.assert_close(x[:, :-1], y[:, :-1])\n\ndef test_tiny_sasrec_fit_and_catalog():\n    torch.set_num_threads(1)\n    model = SASRec(7, k=5, dim=8, epochs=2, batch_size=2).fit([np.array([0, 1, 2]), np.array([2, 3, 4]), np.array([4, 5, 6])])\n    assert len(model.losses) == 2\n    scores = model.score_batch([[0, 1], [3]])\n    assert scores.shape == (2, 7)\n    assert np.isfinite(scores).all()\n', 'tests/test_tensor.py': 'import numpy as np\nimport pytest\nfrom satf_padding.tensor import encode, position_slices, contract, attention_matrix, LASATF\nfrom satf_padding.evaluation import target_rank\n\n@pytest.mark.parametrize("mode", ["zero", "pad", "bos", "start"])\ndef test_sparse_contractions_equal_dense_hankel(mode):\n    histories = [np.array([0, 1]), np.array([2]), np.array([1, 0, 2, 3, 4, 5])]\n    slices = position_slices(histories, 6, 6, mode, amplitude=.7, balance=True)\n    x = np.stack([a.toarray() for a in slices], axis=-1)\n    hankel = np.stack([x[:, :, a:a+4] for a in range(3)], axis=2)\n    a = attention_matrix(3, .8)\n    y = np.einsum("uijs,ja->uias", hankel, a)\n    rng = np.random.default_rng(13)\n    factors = [np.linalg.qr(rng.normal(size=(d, 2)))[0] for d in y.shape]\n    letters = "uial"\n    ranks = "vwbc"\n    for mode_id in range(4):\n        lhs = [letters]\n        operands = [y]\n        out = letters[mode_id]\n        for dim in range(4):\n            if dim != mode_id:\n                lhs.append(letters[dim] + ranks[dim]); operands.append(factors[dim]); out += ranks[dim]\n        expected = np.einsum(",".join(lhs) + "->" + out, *operands).reshape(y.shape[mode_id], -1)\n        np.testing.assert_allclose(contract(slices, factors, a, mode_id), expected, atol=1e-11)\n\ndef test_repeated_pad_and_balancing():\n    ids, p, vals = encode([1, 2], 6, "pad", 7, balance=True)\n    assert sum(ids == 7) == 4\n    assert set(p[ids == 7]) == {0, 1, 2, 3}\n    assert np.sum(vals[ids == 7] ** 2) == pytest.approx(1.)\n\ndef test_full_histories_identical_and_no_spurious_bos_on_shift():\n    expected = encode([0, 1, 2, 3], 4)\n    for mode in ("pad", "bos"):\n        for a, b in zip(expected, encode([0, 1, 2, 3], 4, mode, 4)):\n            np.testing.assert_array_equal(a, b)\n        ids, p, _ = encode([0, 1, 2, 3, 0], 4, mode, 4)\n        assert 4 not in ids[p > 0]\n\ndef test_folding_in_matches_dense_projector():\n    histories = [np.array([0, 1]), np.array([1, 2]), np.array([2, 3, 0]), np.array([3, 1, 4]), np.array([4, 0, 2])]\n    model = LASATF(5, k=6, window=3, rank=2, local_rank=2, sequence_rank=2, mode="pad", iterations=2).fit(histories)\n    h = [0, 1]\n    ids, p, vals = encode(h, 6, "pad", 5)\n    raw = np.zeros((6, 6))\n    for i, pos, v in zip(ids, p, vals):\n        if pos > 0:\n            raw[i, pos-1] += v * model.weights[i]\n    hankel = np.stack([raw[:, a:a+4] for a in range(3)], axis=1)\n    _, v, wl, ws = model.factors\n    pa = np.linalg.solve(model.attention.T, wl) @ wl.T @ model.attention.T\n    ps = ws @ ws.T\n    dense = np.einsum("ij,jab,la,sb->ils", v@v.T, hankel, pa, ps)\n    np.testing.assert_allclose(model.score_batch([h])[0], dense[:5, -1, -1], atol=1e-10)\n    assert model.score_batch([h]).shape == (1, 5)\n    assert np.all(np.diff(model.core_norms) >= -1e-8)\n\ndef test_seen_items_and_ties():\n    assert target_rank([3., 2., 2., 1.], [0], 2) == 2\n    with pytest.raises(ValueError):\n        target_rank([2., 1.], [0], 0)\n', 'configs/ablations.json': '{\n  "name": "mechanism_controls_v1",\n  "datasets": ["ml1m", "beauty", "games"],\n  "seeds": [17, 29, 43],\n  "quantiles": [0.8, 0.9],\n  "k": 50,\n  "tensor": {"rank": 12, "window": 5, "local_rank": 2, "sequence_rank": 2, "decay": 1.0, "scaling": 1.0, "iterations": 3},\n  "models": ["pad_balanced", "start", "zero_scaled", "pad_scaled", "bos_scaled", "pad_scaled_frequency"]\n}\n', 'configs/interim.json': '{\n  "name": "interim_fixed_budget_v1",\n  "datasets": ["ml1m", "beauty", "games"],\n  "seeds": [17, 29, 43],\n  "quantiles": [0.8, 0.9],\n  "k": 50,\n  "tensor": {"rank": 12, "window": 5, "local_rank": 2, "sequence_rank": 2, "decay": 1.0, "scaling": 1.0, "iterations": 3},\n  "sasrec": {"dim": 32, "blocks": 1, "dropout": 0.2, "lr": 0.001, "epochs": 5, "batch_size": 128},\n  "models": ["popular", "zero", "pad", "bos", "sasrec"]\n}\n', 'scripts/summarize.py': '"""Build report tables/figures from measured trial files, never hardcoded scores."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\nLABELS = {"ml1m": "MovieLens-1M", "beauty": "Amazon Beauty", "games": "Amazon Video Games"}\nMODEL_LABELS = {"popular":"MostPopular", "zero":"Zero", "pad":"PAD", "bos":"BOS", "sasrec":"SASRec", "pad_balanced":"PAD / sqrt(q)", "start":"Leftmost marker", "zero_scaled":"Zero, scale=0.2", "pad_scaled":"PAD, scale=0.2", "bos_scaled":"BOS, scale=0.2", "pad_scaled_frequency":"PAD, scale=0.2, freq"}\n\ndef summarize(root):\n    root = Path(root)\n    rows = []\n    for p in sorted(root.glob("*.json")):\n        trial = json.loads(p.read_text())\n        if "model" not in trial:\n            continue\n        rows.append({"dataset":trial["dataset"], "model":trial["model"], "seed":trial["seed"], "train_seconds":trial["fit_seconds"], "peak_rss_mb":trial["peak_rss_mb"], "nnz":trial["nnz"], **trial["test"], "val_ndcg":trial["validation"]["ndcg"]})\n    df = pd.DataFrame(rows)\n    df.to_csv(root/"runs.csv", index=False)\n    summary = df.groupby(["dataset", "model"]).agg(seeds=("seed","count"), n=("n","first"), ndcg=("ndcg","mean"), ndcg_sd=("ndcg","std"), hr=("hr","mean"), val_ndcg=("val_ndcg","mean"), train_seconds=("train_seconds","mean"), peak_rss_mb=("peak_rss_mb","max"), ms_per_user=("ms_per_user","mean"), short_n=("short_n","first"), short_ndcg=("short_ndcg","mean"), full_n=("full_n","first"), full_ndcg=("full_ndcg","mean")).reset_index()\n    summary.to_csv(root/"summary.csv", index=False)\n    for metric, ylabel, filename in [("ndcg","NDCG@10", "quality"), ("train_seconds","Training time, seconds", "training_time"), ("peak_rss_mb","Peak process RSS, MiB", "memory")]:\n        fig, axes = plt.subplots(1, len(summary.dataset.unique()), figsize=(11,3.2), squeeze=False)\n        for ax, ds in zip(axes[0], [x for x in LABELS if x in summary.dataset.unique()]):\n            group = summary[summary.dataset==ds].set_index("model")\n            names = [m for m in MODEL_LABELS if m in group.index]\n            values = group.loc[names,metric]\n            ax.bar(range(len(names)), values, color=["#9ca3af" if m=="popular" else "#93a4b5" if m=="zero" else "#147d92" if m=="pad" else "#e7a13d" if m=="bos" else "#704e99" for m in names])\n            if metric=="ndcg":\n                ax.errorbar(range(len(names)), values, yerr=group.loc[names,"ndcg_sd"].fillna(0), fmt="none", capsize=3, color="#273647")\n            ax.set_xticks(range(len(names)), [MODEL_LABELS[m] for m in names], rotation=45, ha="right", fontsize=8)\n            ax.set_title(LABELS[ds], fontsize=10)\n            ax.set_ylabel(ylabel); ax.spines[["top","right"]].set_visible(False)\n        fig.tight_layout(); fig.savefig(root/(filename+".png"), dpi=180); plt.close(fig)\n    paired = []\n    rng = np.random.default_rng(20261121)\n    for ds in df.dataset.unique():\n        zero_files = sorted(root.glob(f"{ds}_zero_*_test.npz"))\n        if not zero_files:\n            continue\n        for mode in [m for m in ["pad","bos","sasrec"] if m in df.model.unique()]:\n            diffs = []\n            for zp in zero_files:\n                mp = zp.with_name(zp.name.replace("_zero_", f"_{mode}_"))\n                if mp.exists():\n                    with np.load(zp) as a, np.load(mp) as b:\n                        np.testing.assert_array_equal(a["lengths"], b["lengths"])\n                        diffs.append(b["ndcg"] - a["ndcg"])\n            if not diffs:\n                continue\n            per_user = np.mean(diffs, axis=0)\n            bootstrap = [rng.choice(per_user, len(per_user), replace=True).mean() for _ in range(2000)]\n            lo, hi = np.quantile(bootstrap,[.025,.975])\n            paired.append({"dataset":ds,"contrast":mode+"-zero","delta_ndcg":float(per_user.mean()),"ci_user_low":lo,"ci_user_high":hi,"n_users":len(per_user),"n_seeds":len(diffs),"seed_delta_sd":float(np.std(np.mean(diffs,axis=1),ddof=1)) if len(diffs)>1 else None})\n    pd.DataFrame(paired).to_csv(root/"paired.csv", index=False)\n    print(summary.to_string(index=False))\n\nif __name__ == "__main__":\n    p=argparse.ArgumentParser(); p.add_argument("--results",default="results/interim")\n    summarize(p.parse_args().results)\n', 'scripts/verify_results.py': '"""Check completeness, provenance consistency and NPZ-derived metrics."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom satf_padding.experiment import source_hash\n\ndef verify(root, config):\n    root = Path(root); cfg = json.loads(Path(config).read_text())\n    count = 0\n    source_hashes = set()\n    for ds in cfg["datasets"]:\n        stats = json.loads((root/f"{ds}_data.json").read_text())\n        if stats["t1"] >= stats["t2"]:\n            raise ValueError("Invalid temporal cutoffs")\n        for model in cfg["models"]:\n            for seed in (cfg["seeds"][:1] if model == "popular" else cfg["seeds"]):\n                p = root/f"{ds}_{model}_{seed}.json"\n                run = json.loads(p.read_text()); count += 1\n                source_hashes.add(run["source_sha256"])\n                assert run["config"] == cfg\n                assert run["split_sha256"] == stats["split_sha256"]\n                for split in ["validation","test"]:\n                    with np.load(p.with_name(p.stem+"_"+split+".npz")) as a:\n                        assert len(a["ranks"]) == stats[split]["evaluated"]\n                        assert np.isfinite(a["ndcg"]).all()\n                        recomputed = np.where(a["ranks"] <= 10, 1/np.log2(a["ranks"]+1), 0.)\n                        np.testing.assert_allclose(a["ndcg"], recomputed)\n                        assert abs(recomputed.mean()-run[split]["ndcg"]) < 1e-12\n    assert len(source_hashes) == 1, "Mixed implementation versions"\n    assert source_hashes == {source_hash()}, "Recorded source differs from this checkout"\n    print(f"Verified {count} runs in {root}; consistent source and splits")\n    return count\n\nif __name__ == "__main__":\n    p=argparse.ArgumentParser(); p.add_argument("--results",default="results/interim"); p.add_argument("--config",default="configs/interim.json")\n    args=p.parse_args();verify(args.results,args.config)\n'}
SOURCE_MANIFEST = {'pyproject.toml': 'a968f3c45ac3a807d334be478b96050bd75c651d77b8cb4a0cb2571d3ac96127', 'docs/interim/protocol.md': '715cac5a1749bbd91c1ab6a45b9c14457dd988ca033bcf6184092d2d1674e7ee', 'src/satf_padding/__init__.py': '402c8e3fdb1c2c525c90224c9f19c1dd85fd72d2de80faa6588b0f1f459e6dec', 'src/satf_padding/data.py': 'c398003e51a99c4a7f94bd4da0a06ea462befa02080b5e49dc0849de8bf2dd79', 'src/satf_padding/evaluation.py': '4378cc00fc3a40418506ef6c3c05a534d553752199ec3f6cee45369b3a158d42', 'src/satf_padding/experiment.py': '0474849c38aa69f7e26a2798514abfa0c30bee23f0ee314225831a895335f5a1', 'src/satf_padding/sasrec.py': '6a45b5c608c1f9bb0e2e509ccdd31ac74eb0cd94681ffee9a330b6ed19d92fe4', 'src/satf_padding/tensor.py': '7bb2154869699d63dce0cc0909d9c037ac512d9b054d8ca13741fd7888d438e7', 'tests/test_protocol.py': 'a3724376bcb1a22834b4906107d11ef2e24e21355f2cfcba82cea279c9303a03', 'tests/test_sasrec.py': '96512ac60de9dffe10bff17df08c7078298d5a1809efffa9ebf6672c84897626', 'tests/test_tensor.py': '5d237f8e283568ef6d5b6c916c7cf5172ba99d0c067de479f8ea9d7444251eff', 'configs/ablations.json': '2c24bbe7a7156842ba07a334abeccdb9c1e0a8a89a6a474d827ea26802a5ddaa', 'configs/interim.json': '13dd392f6c5502fb157f73255bf5fb1aa8a472d3c9c81cce470f841433ff1d96', 'scripts/summarize.py': '590e21c13b4c6663a697b339213e6a36f29bc17fd9d6c31f3b580f32e78df017', 'scripts/verify_results.py': 'a667b3584552213d6dfac54fad39f91a36a3f5063d8bb3210275e3e3c89a6bb1'}
RAW_SHA256 = {'ml1m': 'a6898adb50b9ca05aa231689da44c217cb524e7ebd39d264c56e2832f2c54e20', 'beauty': '6b4314ec0d74fba455d4ada65cf1313194665797c0f2d33a7c9a516da3968c20', 'games': 'db2a33eba13cc437077e2c1f98cccf656922bf28203585d561c48fb1048184ea'}


In [ ]:
for relative, body in PROJECT_FILES.items():
    assert hashlib.sha256(body.encode()).hexdigest() == SOURCE_MANIFEST[relative]
    dest = PROJECT / relative
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(body, encoding="utf-8")
sys.path.insert(0, str(PROJECT / "src"))
os.environ["PYTHONPATH"] = str(PROJECT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
import importlib
importlib.invalidate_caches()
# При повторном старте настройки перечитываем именно встроенную версию.
for name in list(sys.modules):
    if name == "satf_padding" or name.startswith("satf_padding."):
        del sys.modules[name]
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image, FileLink
from satf_padding.experiment import source_hash
DEVICE = "cuda" if SASREC_DEVICE == "auto" and torch.cuda.is_available() else "cpu"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_ROOT = PROJECT / "runs" / RUN_ID
MAIN = RUN_ROOT / "main"
CONTROLS = RUN_ROOT / "controls"
FIGURES = RUN_ROOT / "figures"
for folder in (MAIN, CONTROLS, FIGURES):
    folder.mkdir(parents=True, exist_ok=True)
CONFIG = json.loads(Path("configs/interim.json").read_text())
DATASETS = CONFIG["datasets"]
LABELS = {"ml1m": "MovieLens-1M", "beauty": "Amazon Beauty", "games": "Amazon Video Games"}
MODEL_LABELS = {"popular": "MostPopular", "zero": "Zero", "pad": "PAD", "bos": "BOS", "sasrec": "SASRec (5 epochs)"}
MODEL_ORDER = list(MODEL_LABELS)
environment = {"run_id": RUN_ID, "python": sys.version, "platform": platform.platform(),
               "processor": platform.processor(), "cpu_count": os.cpu_count(), "threads": 2,
               "sasrec_device": DEVICE, "tensor_device": "cpu", "source_sha256": source_hash(),
               "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None}
(RUN_ROOT / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
(RUN_ROOT / "requirements-resolved.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")
(RUN_ROOT / "source_manifest.json").write_text(json.dumps(SOURCE_MANIFEST, indent=2), encoding="utf-8")

def run_command(arguments):
    """Stream progress in Colab; preserve the full log; stop on failure."""
    with (RUN_ROOT / "execution.log").open("a", encoding="utf-8") as log:
        process = subprocess.Popen([sys.executable, "-u", *map(str, arguments)],
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        result = process.wait()
        if result:
            raise RuntimeError(f"Команда завершилась с кодом {result}: {arguments}")

print("LA-SATF: CPU; SASRec:", DEVICE, "| threads: 2")
print("Результаты нового запуска:", RUN_ROOT)
display(pd.DataFrame([environment]))
print(json.dumps(CONFIG, indent=2))


## 2. Проверка реализации и получение данных

Проверяем повторяющийся PAD, четыре Hankel-свёртки против плотного эталона,
folding-in, совпадение полных историй, исключение служебных объектов,
временное разбиение, ранги и причинную маску SASRec. Ошибка останавливает выполнение.

Протокол: любая оценка — взаимодействие; повтор пары удаляется с сохранением первого
события. Общие временные границы — 80-й и 90-й процентили. Каталог и модели обучаются
только на train. Target — первое событие периода; неизвестный объект не заменяем
более поздним известным. История берётся на момент предсказания; на test доступны
прошлые события validation, без переобучения модели. Оценка по полному каталогу
с исключением всей просмотренной истории и служебных объектов.

In [ ]:
run_command(["-m", "pytest", "-q"])
from satf_padding.data import download, SOURCES
for dataset in DATASETS:
    print(f"Получение {LABELS[dataset]}: {SOURCES[dataset][1]}", flush=True)
    for attempt in range(3):
        try:
            path = download(dataset)
            break
        except Exception:
            if attempt == 2:
                raise
            print("Загрузка прервалась; повтор через 3 секунды", flush=True)
            time.sleep(3)
    actual_sha = hashlib.sha256(path.read_bytes()).hexdigest()
    if actual_sha != RAW_SHA256[dataset]:
        raise ValueError(f"Изменился исходный архив {dataset}: проверьте версию данных")
    print(f"  {path.stat().st_size / 2**20:.1f} MiB; SHA-256 совпадает", flush=True)


## 3. Обучение и оценка всех основных моделей

39 запусков: три кодировки LA-SATF и SASRec по три seed на трёх датасетах,
плюс детерминированный MostPopular по одному разу. Каждое обучение идёт в отдельном
процессе. Сохраняются validation/test-ранги, NDCG@10, HR@10, время, RSS и конфигурация.
После каждой модели появится строка прогресса. Если прервать эту ячейку, её можно
выполнить повторно: уже завершённые запуски пропускаются.

In [ ]:
main_started = time.perf_counter()
run_command(["-m", "satf_padding.experiment", "--config", "configs/interim.json",
             "--output", MAIN, "--device", DEVICE, "--threads", "2"])
main_elapsed = time.perf_counter() - main_started
run_command(["scripts/verify_results.py", "--results", MAIN, "--config", "configs/interim.json"])
run_command(["scripts/summarize.py", "--results", MAIN])
summary = pd.read_csv(MAIN / "summary.csv")
runs = pd.read_csv(MAIN / "runs.csv")
paired = pd.read_csv(MAIN / "paired.csv")
stats = {ds: json.loads((MAIN / f"{ds}_data.json").read_text()) for ds in DATASETS}
data_rows = []
for ds, s in stats.items():
    data_rows.append({"Датасет": LABELS[ds], "События": s["events"],
                      "Train": s["train_events"], "Каталог train": s["catalog"],
                      "Validation": s["validation"]["evaluated"], "Test": s["test"]["evaluated"],
                      "Cold targets test": s["test"]["cold_target"],
                      "Нет истории test": s["test"]["empty_known_history"],
                      "t1 UTC": pd.to_datetime(s["t1"], unit="s", utc=True),
                      "t2 UTC": pd.to_datetime(s["t2"], unit="s", utc=True)})
data_table = pd.DataFrame(data_rows)
display(data_table)
data_table.to_csv(RUN_ROOT / "dataset_statistics.csv", index=False)
print(f"Основные запуски с подготовкой данных: {main_elapsed / 60:.1f} мин.")


## 4. H1: качество Zero / PAD / BOS и сравнение с бейзлайнами

Основная метрика — **NDCG@10**, дополнительно **HR@10**. Для одного target HR@10
совпадает с Recall@10. В таблице среднее ± SD по трём seed; для MostPopular один запуск.
Отрезки на графиках — SD, не доверительный интервал. Validation показывается отдельно.

In [ ]:
def measured_table(frame, metric, deviation=None, order=None):
    order = order or list(dict.fromkeys(frame["model"]))
    table = pd.DataFrame(index=order, columns=[LABELS[ds] for ds in DATASETS])
    for row in frame.to_dict("records"):
        value = f"{row[metric]:.5f}"
        if deviation and pd.notna(row.get(deviation)):
            value += f" ± {row[deviation]:.5f}"
        table.loc[row["model"], LABELS[row["dataset"]]] = value
    return table.rename(index=MODEL_LABELS)

display(Markdown("**Test NDCG@10, mean ± SD**"))
display(measured_table(summary, "ndcg", "ndcg_sd", MODEL_ORDER))
display(Markdown("**Test HR@10, mean**"))
display(measured_table(summary, "hr", order=MODEL_ORDER))
display(Markdown("**Validation NDCG@10, mean**"))
display(measured_table(summary, "val_ndcg", order=MODEL_ORDER))
display(Image(filename=str(MAIN / "quality.png")))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.7))
colors = ["#9ca3af", "#93a4b5", "#147d92", "#e7a13d", "#704e99"]
for ax, ds in zip(axes, DATASETS):
    group = runs[runs.dataset == ds].groupby("model").hr.agg(["mean", "std"]).loc[MODEL_ORDER]
    ax.bar(range(5), group["mean"], color=colors, yerr=group["std"].fillna(0), capsize=3)
    ax.set_xticks(range(5), [MODEL_LABELS[m] for m in MODEL_ORDER], rotation=35, ha="right")
    ax.set_title(LABELS[ds]); ax.set_ylabel("Test HR@10")
fig.tight_layout(); fig.savefig(FIGURES / "hr10.png", dpi=160); plt.show(); plt.close(fig)


Парное сравнение: для каждого пользователя усредняем метрику по seed, затем
2000 раз пересэмплируем пользователей. **95% интервалы условны на трёх обучениях**:
они не охватывают всю неопределённость обучения и не скорректированы за множественные
сравнения. Положительное среднее само по себе не доказывает устойчивое улучшение.

In [ ]:
display(paired[paired.contrast.isin(["pad-zero", "bos-zero"])])
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, ds in zip(axes, DATASETS):
    group = paired[(paired.dataset == ds) & paired.contrast.isin(["pad-zero", "bos-zero"])]
    for i, row in enumerate(group.itertuples()):
        ax.plot([row.ci_user_low, row.ci_user_high], [i, i], color="#147d92", linewidth=3)
        ax.scatter([row.delta_ndcg], [i], color="#263649", zorder=3)
    ax.axvline(0, linestyle="--", color="#999999")
    ax.set_yticks(range(len(group)), group.contrast)
    ax.set_xlabel("Delta NDCG@10 and conditional 95% CI")
    ax.set_title(LABELS[ds]); ax.set_ylim(-.5, 1.5)
fig.tight_layout(); fig.savefig(FIGURES / "paired_intervals.png", dpi=160); plt.show(); plt.close(fig)


## 5. H2: стоимость BOS относительно PAD

Сопоставляем только три тензорные кодировки на одном CPU. `fit` включает подготовку
представления и проектора, но не скачивание/чтение данных. Оценка включает вычисление
scores, маскирование и ранги. RSS — память **всего процесса** при fit+eval,
измеряемая каждые 20 мс; это не размер модели и не VRAM.
Порог эквивалентности качества заранее не задан, поэтому «сопоставимое качество»
нельзя формально подтвердить только по близости средних.

In [ ]:
cost = summary[summary.model.isin(["zero", "pad", "bos"])][
    ["dataset", "model", "ndcg", "train_seconds", "ms_per_user", "peak_rss_mb"]]
display(cost)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.7))
for ax, metric, label in zip(axes, ["train_seconds", "ms_per_user", "peak_rss_mb"],
                             ["Fit, seconds (mean)", "Evaluation, ms/user (mean)", "Process peak RSS, MiB (max)"]):
    for j, model in enumerate(["zero", "pad", "bos"]):
        group = cost[cost.model == model].set_index("dataset").loc[DATASETS]
        ax.bar(np.arange(3) + (j-1)*.24, group[metric], width=.24,
               label=MODEL_LABELS[model], color=colors[j+1])
    ax.set_xticks(range(3), ["ML-1M", "Beauty", "Games"])
    ax.set_ylabel(label); ax.legend()
fig.tight_layout(); fig.savefig(FIGURES / "tensor_cost.png", dpi=160); plt.show(); plt.close(fig)


## 6. H3: эффект по длине доступной истории

Длина — число известных объектов **на момент предсказания**, обрезанное до K=50.
Одинаковые пользователи входят в каждый сравниваемый срез. При n≥50 входные
кодировки совпадают, однако отдельно обученные модели могут выдавать разные scores.
Над каждым срезом указано число пользователей; редкие полные истории Amazon
не позволяют делать надёжные выводы об отсутствии эффекта.

In [ ]:
history_rows = []
bins = [("1–5", 1, 5), ("6–20", 6, 20), ("21–49", 21, 49), ("50+", 50, 50)]
for ds in DATASETS:
    for model in ["zero", "pad", "bos"]:
        for seed in CONFIG["seeds"]:
            with np.load(MAIN / f"{ds}_{model}_{seed}_test.npz") as a:
                for label, lo, hi in bins:
                    mask = (a["lengths"] >= lo) & (a["lengths"] <= hi)
                    history_rows.append({"dataset": ds, "model": model, "seed": seed,
                        "history": label, "n": int(mask.sum()),
                        "ndcg": float(a["ndcg"][mask].mean()) if mask.any() else np.nan})
history = pd.DataFrame(history_rows).groupby(["dataset", "model", "history"], sort=False).agg(
    n=("n", "first"), ndcg=("ndcg", "mean"), sd=("ndcg", "std")).reset_index()
history.to_csv(RUN_ROOT / "history_slices.csv", index=False)
display(history)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.7))
for ax, ds in zip(axes, DATASETS):
    for model, color in zip(["zero", "pad", "bos"], colors[1:4]):
        group = history[(history.dataset == ds) & (history.model == model)].set_index("history").loc[[b[0] for b in bins]]
        ax.errorbar(range(4), group.ndcg, yerr=group.sd.fillna(0), marker="o", capsize=3,
                    label=MODEL_LABELS[model], color=color)
    ax.set_xticks(range(4), [f"{b[0]}\nn={int(group.loc[b[0], 'n'])}" for b in bins])
    ax.set_ylabel("Test NDCG@10, mean ± SD"); ax.set_title(LABELS[ds]); ax.legend()
fig.tight_layout(); fig.savefig(FIGURES / "history_slices.png", dpi=160); plt.show(); plt.close(fig)


## 7. H4: контроль веса служебных записей и нормализации

Ещё **54 обучения**: шесть вариантов × три датасета × три seed.

- `pad_balanced`: вес каждого PAD = 1/√q; уравнивает энергию служебных записей
  до Hankel, но не полный вклад истории после преобразования.
- `start`: один маркер в крайней левой позиции. Он удаляется сдвигом при предсказании,
  поэтому это контроль влияния служебной записи через обучение.
- `zero_scaled`, `pad_scaled`, `bos_scaled`: нормализация реальных объектов, scaling=0.2.
- `pad_scaled_frequency`: дополнительно нормируется PAD по его частоте в train.

Сравнения помогают обнаружить чувствительность к весам и нормировке. Они не изолируют
все причины: отдельная нормализация вклада историй после Hankel остаётся следующим шагом.

In [ ]:
controls_started = time.perf_counter()
run_command(["-m", "satf_padding.experiment", "--config", "configs/ablations.json",
             "--output", CONTROLS, "--device", "cpu", "--threads", "2"])
controls_elapsed = time.perf_counter() - controls_started
run_command(["scripts/verify_results.py", "--results", CONTROLS, "--config", "configs/ablations.json"])
run_command(["scripts/summarize.py", "--results", CONTROLS])
ablations = pd.read_csv(CONTROLS / "summary.csv")
combined = pd.concat([summary[summary.model.isin(["zero", "pad", "bos"])], ablations], ignore_index=True)
ablation_order = ["zero", "pad", "bos", "pad_balanced", "start", "zero_scaled", "pad_scaled", "bos_scaled", "pad_scaled_frequency"]
display(measured_table(combined, "ndcg", "ndcg_sd", ablation_order))
combined.to_csv(RUN_ROOT / "mechanism_comparison.csv", index=False)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, ds in zip(axes, DATASETS):
    group = combined[combined.dataset == ds].set_index("model").loc[ablation_order]
    ax.barh(range(len(group)), group.ndcg, xerr=group.ndcg_sd.fillna(0), color="#147d92", capsize=3)
    ax.set_yticks(range(len(group)), ablation_order, fontsize=8); ax.invert_yaxis()
    ax.set_xlabel("Test NDCG@10, mean ± SD"); ax.set_title(LABELS[ds])
fig.tight_layout(); fig.savefig(FIGURES / "mechanism_controls.png", dpi=160); plt.show(); plt.close(fig)
print(f"Контрольные запуски с подготовкой данных: {controls_elapsed / 60:.1f} мин.")


## 8. Диагностика обучения SASRec

Показываем train-loss каждой из пяти эпох и качество validation/test.
Уменьшение train-loss не доказывает сходимость или оптимальность модели.
Этот короткий запуск — пилотный бейзлайн; он не позволяет заявить превосходство
над настроенным SASRec. Для дальнейшего сравнения нужны кривые validation
по эпохам, ранняя остановка и равный бюджет подбора конфигураций.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, ds in zip(axes, DATASETS):
    for seed in CONFIG["seeds"]:
        trial = json.loads((MAIN / f"{ds}_sasrec_{seed}.json").read_text())
        ax.plot(range(1, len(trial["losses"])+1), trial["losses"], marker="o", label=f"seed {seed}")
    ax.set_title(LABELS[ds]); ax.set_xlabel("Epoch"); ax.set_ylabel("Train sampled BCE"); ax.legend()
fig.tight_layout(); fig.savefig(FIGURES / "sasrec_training.png", dpi=160); plt.show(); plt.close(fig)
display(summary[summary.model == "sasrec"][["dataset", "val_ndcg", "ndcg", "ndcg_sd", "hr", "train_seconds"]])


## 9. Выводы по вашему запуску

Следующая ячейка подставляет фактически полученные числа. Статус гипотез
не задан заранее. Положительные пилотные контрасты не считаются окончательным
подтверждением общего выигрыша метода.

In [ ]:
lookup = summary.set_index(["dataset", "model"])
control_lookup = ablations.set_index(["dataset", "model"])
lines = ["# Выводы текущего запуска", "", "## H1 — качество относительно Zero"]
for ds in DATASETS:
    for model in ["pad", "bos"]:
        p = paired[(paired.dataset == ds) & (paired.contrast == f"{model}-zero")].iloc[0]
        status = "целиком выше нуля" if p.ci_user_low > 0 else "целиком ниже нуля" if p.ci_user_high < 0 else "включает ноль"
        lines.append(f"- {LABELS[ds]}, {MODEL_LABELS[model]} − Zero: ΔNDCG@10 = {p.delta_ndcg:+.5f}; "
                     f"условный 95% CI [{p.ci_user_low:+.5f}; {p.ci_user_high:+.5f}], {status}.")
lines += ["", "Интервалы описывают вариацию пользователей при этих трёх seed; поправки за множественные сравнения нет.",
          "", "## H2 — стоимость BOS относительно PAD"]
for ds in DATASETS:
    p, b = lookup.loc[(ds, "pad")], lookup.loc[(ds, "bos")]
    lines.append(f"- {LABELS[ds]}: fit BOS/PAD = {b.train_seconds/p.train_seconds:.2f}×; "
                 f"peak RSS BOS/PAD = {b.peak_rss_mb/p.peak_rss_mb:.2f}×; "
                 f"ΔNDCG@10 (BOS − PAD) = {b.ndcg-p.ndcg:+.5f}.")
lines += ["", "Отношение меньше 1 означает меньшую измеренную стоимость BOS. "
          "Эквивалентность качества не проверена: заранее заданного допуска нет.",
          "", "## H3 — длина истории"]
for ds in DATASETS:
    z, p = lookup.loc[(ds, "zero")], lookup.loc[(ds, "pad")]
    lines.append(f"- {LABELS[ds]}: PAD − Zero на коротких историях = {p.short_ndcg-z.short_ndcg:+.5f} "
                 f"(n={int(z.short_n)}); на полных = {p.full_ndcg-z.full_ndcg:+.5f} (n={int(z.full_n)}).")
    if z.full_n < 100:
        lines.append("  Полных историй меньше 100: оценка этого среза особенно ненадёжна.")
lines += ["", "Разница средних по срезам — описательный результат; отдельный тест взаимодействия длины и кодировки не выполнен.",
          "", "## H4 — служебные наблюдения и нормировка"]
for ds in DATASETS:
    p, z = lookup.loc[(ds, "pad")], lookup.loc[(ds, "zero")]
    c = control_lookup.loc[ds]
    lines.append(f"- {LABELS[ds]}: balanced PAD − PAD = {c.loc['pad_balanced', 'ndcg']-p.ndcg:+.5f}; "
                 f"левый маркер − Zero = {c.loc['start', 'ndcg']-z.ndcg:+.5f}; "
                 f"частотная нормировка PAD − единичный вес PAD при scaling=0.2 = "
                 f"{c.loc['pad_scaled_frequency', 'ndcg']-c.loc['pad_scaled', 'ndcg']:+.5f}.")
lines += ["", "Эти изменения показывают чувствительность к выбранным вмешательствам. "
          "Отделить эффект самой границы от изменения задачи аппроксимации пока нельзя.",
          "", "## Ограничения и следующий этап", "",
          "- Бюджет фиксирован и мал: 3 итерации HOOI, 5 эпох SASRec, без подбора гиперпараметров.",
          "- Дальше: одинаковый бюджет настройки по validation, SASRec до validation-плато, больше seed, K, окон и временных границ.",
          "- Для механизма: уравнять вклад историй после Hankel и отдельно отключать служебный вклад при инференсе одной обученной модели.",
          "- Warm-item протокол исключает cold targets; Amazon 5-core ограничивает популяцию по всему исходному периоду.",
          "- Уже просмотренный test нельзя использовать как независимое подтверждение новых гипотез; нужны заранее выделенные новые временные интервалы."]
conclusion = "\n".join(lines)
(RUN_ROOT / "conclusions.md").write_text(conclusion, encoding="utf-8")
display(Markdown(conclusion))


## 10. Сохранение результатов

Архив содержит JSON и NPZ всех 93 запусков, CSV, графики, выводы, конфигурации,
использованные исходники и версии библиотек. Сырые датасеты и пользовательские ID
в архив не входят. При завершении Colab-сессии рабочая папка может исчезнуть:
сохраните архив. При открытии локально появится ссылка на него.

In [ ]:
from satf_padding.experiment import source_hash as current_source_hash
assert current_source_hash() == environment["source_sha256"]
main_count = len(runs)
control_count = len(pd.read_csv(CONTROLS / "runs.csv"))
assert main_count == 39 and control_count == 54, (main_count, control_count)
elapsed = time.perf_counter() - NOTEBOOK_STARTED
(RUN_ROOT / "timing.json").write_text(json.dumps({"total_wall_seconds": elapsed,
    "main_command_wall_seconds": main_elapsed, "controls_command_wall_seconds": controls_elapsed,
    "main_runs": main_count, "control_runs": control_count}, indent=2), encoding="utf-8")
archive = PROJECT / f"satf_results_{RUN_ID}.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for path in sorted(RUN_ROOT.rglob("*")):
        if path.is_file():
            z.write(path, "results/" + str(path.relative_to(RUN_ROOT)))
    for relative in PROJECT_FILES:
        z.write(PROJECT / relative, "project/" + relative)
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
print(f"Готово: {main_count} основных + {control_count} контрольных обучений.")
print(f"Общее время, включая скачивание/подготовку/графики: {elapsed / 60:.1f} мин.")
print(f"Архив: {archive.name} ({archive.stat().st_size / 2**20:.2f} MiB)")
try:
    from google.colab import files
except ImportError:
    display(FileLink(str(archive.relative_to(PROJECT))))
else:
    files.download(str(archive))


## Источники и воспроизводимость

- [LA-SATF: Frolov & Oseledets, IEEE Access, 2023](https://doi.org/10.1109/ACCESS.2023.3234863).
- [SASRec: Kang & McAuley, ICDM 2018](https://arxiv.org/abs/1808.09781).
- [MovieLens-1M, GroupLens](https://grouplens.org/datasets/movielens/1m/).
- [Amazon 2014, McAuley Lab](https://cseweb.ucsd.edu/~jmcauley/datasets/amazon/links.html).

Собственные реализации по математическим постановкам; точное воспроизведение
чисел оригинальных статей не заявляется. SHA-256 исходных данных и кода проверяются.
Отличия библиотек, аппаратуры и численных алгоритмов могут менять результаты;
в каждом запуске сохраняется его окружение. Публикация проекта этим ноутбуком не выполняется.